# Session 23 — Automated ML Deployment using BentoML and Docker

**Goal:** package and deploy a model with [BentoML](https://www.bentoml.com/), a
framework purpose-built for ML serving — compare its "save model, define a service,
build a bento" workflow against the hand-written Flask/Dockerfile approach from
Session 6.

## BentoML vs. hand-rolled Flask + Dockerfile (Session 6)

Session 6 wrote the Flask app, `Dockerfile`, and `requirements.txt` by hand. BentoML
generates all of that from a much smaller amount of code: a "Bento" bundles the
model, its runtime dependencies, and a versioned API definition together, and
`bentoml containerize` produces a production-ready Docker image without you writing
a Dockerfile at all.

## Prerequisites

```bash
pip install bentoml
```
Model-saving and service-definition code below runs locally. Building/running the
final Docker image (Step 5) needs a local Docker daemon, same caveat as Session 6 —
shown as the exact commands to run, not executed in this sandbox.

In [ ]:
import bentoml
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

print("bentoml version:", bentoml.__version__)

## Step 1 — Train a model, as usual

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)
print(f"train accuracy: {model.score(X_train, y_train):.4f}")
print(f"test accuracy:  {model.score(X_test, y_test):.4f}")

## Step 2 — Save the model into BentoML's model store

`bentoml.sklearn.save_model` does what `mlflow.sklearn.log_model` (Session 1) does
for tracking, but for the specific purpose of serving: it saves the model plus
signature metadata (expected input/output shapes) into a local, versioned store.

In [ ]:
saved_model = bentoml.sklearn.save_model("iris_classifier", model)
print(f"Saved model: {saved_model.tag}")

for m in bentoml.models.list():
    print(f"  {m.tag}  created {m.creation_time}")

## Step 3 — Define the service

A `service.py` file (written here, would normally live in your project) declares the
API surface — much shorter than Session 6's hand-written Flask app, since BentoML
handles request parsing, batching, and OpenAPI docs generation for you.

In [ ]:
service_code = '''\
import bentoml
import numpy as np
from bentoml.io import NumpyNdarray, JSON

iris_runner = bentoml.sklearn.get("iris_classifier:latest").to_runner()

svc = bentoml.Service("iris_classifier_service", runners=[iris_runner])

@svc.api(input=NumpyNdarray(), output=JSON())
def predict(input_array: np.ndarray) -> dict:
    result = iris_runner.predict.run(input_array)
    return {"predictions": result.tolist()}
'''
with open("service.py", "w") as f:
    f.write(service_code)
print(service_code)

## Step 4 — Serve it locally and test

`bentoml serve` starts a local dev server (run from a terminal, not in-notebook);
the runner API used above can also be called directly in-process for a quick sanity
check without starting a server at all.

In [ ]:
import numpy as np

runner = saved_model.to_runner()
runner.init_local()  # in-process init, for local testing only -- not used in production

sample = np.array(X_test[:3])
predictions = runner.predict.run(sample)
print("Direct runner predictions:", predictions)
print("\nTo serve over HTTP: bentoml serve service:svc --reload")

## Step 5 — Build a Bento and containerize it

A **Bento** is BentoML's packaged unit (model + service code + a `bentofile.yaml`
listing dependencies) — `bentoml build` creates it, `bentoml containerize` turns it
into a Docker image, without you writing a Dockerfile.

In [ ]:
bentofile_yaml = '''\
service: "service:svc"
labels:
  owner: mlops-skilling-course
  stage: dev
include:
  - "service.py"
python:
  packages:
    - scikit-learn
    - numpy
'''
with open("bentofile.yaml", "w") as f:
    f.write(bentofile_yaml)
print(bentofile_yaml)

build_and_containerize = '''\
bentoml build
bentoml containerize iris_classifier_service:latest -t iris-classifier-bento:v1

docker run -p 3000:3000 iris-classifier-bento:v1
# Then: curl -X POST http://localhost:3000/predict -H "Content-Type: application/json" -d "[[5.1,3.5,1.4,0.2]]"
'''
print(build_and_containerize)

## Step 6 — When to reach for BentoML vs. Session 6's manual approach

| | Manual Flask + Dockerfile (Session 6) | BentoML |
|---|---|---|
| Control over every line | Full | Less (framework conventions) |
| Boilerplate | You write it all | Generated from `save_model` + `@svc.api` |
| Multi-model / multi-runner services | Manual wiring | Built-in (`runners=[...]`) |
| Adaptive batching for throughput | Manual | Built-in |
| Best fit | Learning the fundamentals, unusual serving needs | Standardizing serving across many models/teams |

## What to try next

* Add a second runner (e.g. a preprocessing step) to the same service, and see how
  BentoML's `runners=[...]` list composes multiple models into one API.
* Push the built Bento to Yatai (BentoML's model deployment platform) or straight to
  a Kubernetes cluster using `bentoml deployment create`, comparable to Session 6's
  `kubectl apply`.
* Compare cold-start latency and throughput against the Flask/FastAPI apps from
  Sessions 6-7 under load (`locust` or `hey`), since BentoML's adaptive batching can
  meaningfully change both.